In [ ]:
import os,gzip
import pandas as pd
import scanpy as sc
import numpy as np
import scipy.sparse as sp
from scipy.io import mmwrite
import re

In [ ]:
import sys
sys.path.append('/Volumes/B/Projects/2025Project/SpatialAnalysis/Code')
import SpatialPyALab as lab  # type: ignore

In [ ]:
root_dir = "/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC"
os.chdir(root_dir) 

In [ ]:
DATA_Dir = '/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC/Data/ST/Samples/'

In [ ]:
patient_info = pd.read_csv("Table/Patient_Info.csv")
df = patient_info.dropna(subset=["SampleName", "SampleID"]).copy()
samples = df["SampleName"].drop_duplicates().tolist()


In [ ]:
def command_A(spot_adata,coords_key='spatial_manual_fine', hex_edge=36):

   mu_scale = spot_adata.uns['mu_scale'][coords_key]

   mesh_adata,orig_idx,new_idx = lab.mesh.create_meshed_adata(spot_adata,
                                                              coords_key=coords_key,
                                                              px_by_um = mu_scale, 
                                                              spot_diameter_um = hex_edge)
   
   mesh_adata.obsm['spatial_um'] = mesh_adata.obsm['spatial'] / mu_scale
   mesh_adata = mesh_adata[mesh_adata.obs.index!= 0]

   spot_to_hex = lab.annotation.spot_to_hex(mesh_adata)
   df = spot_adata.obs[['barcode', 'CellType','GLMPCA_leiden']]
   
   label_df = lab.annotation.add_ficture_result(spot_to_other=spot_to_hex, 
                                                spot_id='Spot_ID',
                                                segment_id='Hex_ID',
                                                ficture_anno=df, 
                                                barcode_col = 'barcode',
                                                threshold=0.5,
                                                label_cols=['CellType','GLMPCA_leiden'])
   
   mesh_adata.obs['Hex_ID'] = mesh_adata.obs.index.astype(str)
   mesh_adata.obs = mesh_adata.obs.merge(label_df, on="Hex_ID", how="left")

   mesh_adata.obs['Hex_ID'] = 'hex_' + mesh_adata.obs['Hex_ID'].astype(str)
   mesh_adata.obs_names = mesh_adata.obs['Hex_ID'].astype(str)
   mesh_adata.obs.drop(columns=['Hex_ID'], inplace=True)

   return mesh_adata, spot_to_hex

In [ ]:
import multiprocessing as mp
mp.set_start_method("fork", force=True)

In [ ]:
import os
import scanpy as sc
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_one(sid):
    data_dir1 = os.path.join(DATA_Dir, sid)
    h5ad_file = os.path.join(data_dir1, f"ficture/{sid}_anno_ficture.h5ad")
    out_dir = os.path.join(data_dir1, "hexagon")
    os.makedirs(out_dir, exist_ok=True)

    out_h5ad = os.path.join(out_dir, "to_inferCNV_36um.h5ad")
    out_tsv  = os.path.join(out_dir, "spot_to_hex.tsv")

    if not os.path.exists(h5ad_file):
        return sid, None, None, f"missing {h5ad_file}"

    # 可尝试 backed="r"
    spot_adata = sc.read_h5ad(h5ad_file)
    spot_adata.obs["CellType"] = spot_adata.obs["CellType_ficture"]

    adata_s, spot_to_hex = command_A(spot_adata, hex_edge=36)

    adata_s.obs["sample_id"] = sid
    adata_s.obs_names = sid + "_" + adata_s.obs_names.astype(str)

    spot_to_hex = spot_to_hex.copy()
    spot_to_hex["SampleID"] = sid
    spot_to_hex["Hex_ID"]   = sid + "_hex_" + spot_to_hex["Hex_ID"].astype(str)
    spot_to_hex["Spot_ID"]  = sid + "_" + spot_to_hex["Spot_ID"].astype(str)

    adata_s.write(out_h5ad)
    spot_to_hex.to_csv(out_tsv, sep="\t", index=False)

    return sid, out_h5ad, out_tsv, "ok"


max_workers = 16  
out_h5ads, out_tsvs = [], []

with ProcessPoolExecutor(max_workers=max_workers) as ex:
    futs = [ex.submit(process_one, sid) for sid in samples]
    for fu in as_completed(futs):
        sid, p_h5ad, p_tsv, msg = fu.result()
        print(f"[{sid}] {msg}")
        if p_h5ad: out_h5ads.append(p_h5ad)
        if p_tsv:  out_tsvs.append(p_tsv)


merge_adatas = [sc.read_h5ad(p) for p in out_h5ads]

In [ ]:
adata_all = sc.concat(merge_adatas,join="outer",label="batch",fill_value=0)
adata_all

In [ ]:
adata_all.obs["barcode"] = adata_all.obs.index.astype(str)
adata_all.obs["cell_id"] = adata_all.obs.index.astype(str)

In [ ]:
adata_all.obs = adata_all.obs.merge(patient_info,left_on="sample_id",right_on="SampleName",how="left")

In [ ]:
pat = re.compile(r'^(ENSG|ENST|ENSP|ENS)\d+(\.\d+)?$') 
mask = ~adata_all.var_names.astype(str).str.match(pat)
adata_all = adata_all[:, mask].copy()
adata_all

In [ ]:
adata_all.write(os.path.join(root_dir,'Data/ST/inferCNV/RawData/1_Hex_Level_samples_merged_original.h5ad'))

In [ ]:
 
h5ad_path = "Data/ST/inferCNV/RawData/1_Hex_Level_samples_merged_original.h5ad"
adata = sc.read_h5ad(h5ad_path)
adata

In [ ]:
coords = pd.DataFrame(adata.obsm['spatial_um'], index=adata.obs['barcode'], columns=["x", "y"])
coords["SampleID"] = adata.obs["SampleID"].astype(str).to_numpy()
coords["PatientID"] = adata.obs["PatientID"].astype(str).to_numpy()
coords.index.name = "barcode"
coords

In [ ]:
coords.to_csv("Data/ST/inferCNV/RawData/create_Seurat_data/coordinates.tsv", sep="\t")

In [ ]:
outdir = os.path.join(root_dir,'Data/ST/inferCNV/RawData/create_Seurat_data')
os.makedirs(outdir, exist_ok=True)

In [ ]:
X = adata.X
var_names = adata.var_names

if not sp.issparse(X):
    X = sp.csr_matrix(np.asarray(X))
X = X.tocsr().astype(np.float32)
if X.nnz > 0:
    X.data = np.clip(X.data, 0, None)

mtx_gz = os.path.join(outdir, "counts.mtx.gz")
with gzip.open(mtx_gz, "wb", compresslevel=1) as f:
    mmwrite(f, X)   

pd.Series(var_names, name="gene").to_csv(
    os.path.join(outdir, "genes.tsv"), sep="\t", index=False, header=False
)
barcodes = adata.obs["barcode"].astype(str).values  
pd.Series(barcodes).to_csv(
    os.path.join(outdir, "barcodes.tsv"),
    sep="\t", index=False, header=False
)
adata.obs.to_csv(os.path.join(outdir, "metadata.csv"))
 
